In [ ]:
# ============================================================
# Module 0. Imports, random seed and publication-style settings
# ============================================================
import os
import math
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy import stats
from scipy.stats import shapiro, jarque_bera, skew, kurtosis, spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import networkx as nx

try:
    import lingam
    LINGAM_AVAILABLE = True
except Exception as e:
    LINGAM_AVAILABLE = False
    print("[Warning] lingam is not available. Please install it before running DirectLiNGAM diagnostics:", e)

try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    from statsmodels.stats.stattools import durbin_watson
    STATSMODELS_AVAILABLE = True
except Exception as e:
    STATSMODELS_AVAILABLE = False
    print("[Warning] statsmodels is not available. Temporal residual diagnostics will be skipped:", e)

try:
    from econml.dml import CausalForestDML
    ECONML_AVAILABLE = True
except Exception as e:
    ECONML_AVAILABLE = False
    print("[Warning] econml is not available. CausalForestDML functions will not run until econml is installed:", e)

warnings.filterwarnings("ignore")

SEED = 66
random.seed(SEED)
np.random.seed(SEED)

# ---------- High-level SCI figure style ----------
mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
    "axes.linewidth": 1.0,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "savefig.dpi": 600,
    "figure.dpi": 150,
})

PALETTE = {
    "blue": "#2A6FBB",
    "cyan": "#45A6B8",
    "teal": "#2F8F83",
    "orange": "#E5863D",
    "red": "#B94E48",
    "purple": "#7A5BA6",
    "grey": "#6F7C85",
    "light_grey": "#EAECEF",
    "dark": "#243447"
}

OUT_DIR = Path("diagnostic_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("Environment ready.")

In [ ]:
# ============================================================
# Module 1. User-defined data path, variables and column mapping
# ============================================================
DATA_PATH = ""  

TARGET = "n"



In [ ]:
# ============================================================
# Module 2. Data loading and cleaning
# ============================================================
def load_and_prepare_data(data_path=DATA_PATH,
                          rename_map=RENAME_MAP,
                          selected_features=SELECTED_FEATURES):
    """Load data, standardise column names, keep selected variables, and coerce numeric values."""
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"Cannot find {data_path}. Please place the CSV file in the notebook directory or modify DATA_PATH."
        )
    df = pd.read_csv(data_path)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns=rename_map)
    df.columns = [str(c).strip() for c in df.columns]

    selected_features = [c.strip() for c in selected_features]
    missing_cols = [c for c in selected_features if c not in df.columns]
    if missing_cols:
        raise ValueError(f"These selected variables are missing from the dataset: {missing_cols}")

    df = df[selected_features].copy()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Median imputation is used to prevent DirectLiNGAM / DML from failing because of missing values. Replace this with KNN imputation here if it is used in the manuscript.
    imputer = SimpleImputer(strategy="median")
    df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns, index=df.index)

    # Remove fully constant columns.
    nunique = df_imputed.nunique(dropna=True)
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        print("Dropped constant columns:", constant_cols)
        df_imputed = df_imputed.drop(columns=constant_cols)

    print(f"Data shape after preparation: {df_imputed.shape}")
    display(df_imputed.head())
    return df_imputed


data = load_and_prepare_data()
analysis_vars = [c for c in data.columns if c != TARGET]
print("Analysis variables:", len(analysis_vars))

In [ ]:
# ============================================================
# Module 3. Non-Gaussianity diagnostics
# ============================================================
def non_gaussianity_diagnostics(df, variables=None, alpha=0.05):
    if variables is None:
        variables = df.columns.tolist()
    rows = []
    for col in variables:
        x = df[col].dropna().values
        if len(x) < 8:
            continue
        # Shapiro works for n <= 5000. For larger n, sample 5000.
        x_shapiro = x if len(x) <= 5000 else np.random.default_rng(SEED).choice(x, size=5000, replace=False)
        sw_stat, sw_p = shapiro(x_shapiro)
        jb_stat, jb_p = jarque_bera(x)
        rows.append({
            "Variable": col,
            "N": len(x),
            "Skewness": skew(x, nan_policy="omit"),
            "Kurtosis(excess)": kurtosis(x, fisher=True, nan_policy="omit"),
            "Shapiro-Wilk p": sw_p,
            "Jarque-Bera p": jb_p,
            "Non-Gaussian by either test": (sw_p < alpha) or (jb_p < alpha)
        })
    result = pd.DataFrame(rows).sort_values("Jarque-Bera p")
    result.to_csv(OUT_DIR / "Table_DirectLiNGAM_non_Gaussianity_diagnostics.csv", index=False)
    return result

ng_table = non_gaussianity_diagnostics(data)
display(ng_table)

# Plot: skewness-kurtosis diagnostic map
def plot_skew_kurtosis_map(result):
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    x = result["Skewness"]
    y = result["Kurtosis(excess)"]
    sig = result["Non-Gaussian by either test"]
    ax.scatter(x[~sig], y[~sig], s=70, c=PALETTE["grey"], edgecolor="white", linewidth=0.8, alpha=0.85, label="Not rejected")
    ax.scatter(x[sig], y[sig], s=90, c=PALETTE["orange"], edgecolor="white", linewidth=0.8, alpha=0.95, label="Non-Gaussian")
    for _, row in result.iterrows():
        if abs(row["Skewness"]) > 1.0 or abs(row["Kurtosis(excess)"]) > 1.5:
            ax.text(row["Skewness"], row["Kurtosis(excess)"], row["Variable"], fontsize=8, alpha=0.85)
    ax.axvline(0, color="#B0B7C3", lw=1.0, ls="--")
    ax.axhline(0, color="#B0B7C3", lw=1.0, ls="--")
    ax.set_xlabel("Skewness")
    ax.set_ylabel("Excess kurtosis")
    ax.set_title("Distributional diagnostic for DirectLiNGAM non-Gaussianity assumption")
    ax.grid(True, color="#E9EDF2", linewidth=0.8)
    ax.legend(frameon=True, edgecolor="#D0D5DD")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DirectLiNGAM_non_Gaussianity_skew_kurtosis.png", bbox_inches="tight")
    plt.show()

plot_skew_kurtosis_map(ng_table)

In [ ]:
# ============================================================
# Module 4. DirectLiNGAM fitting, edge extraction and visualisation
# ============================================================
def fit_direct_lingam(df, threshold=0.05, random_state=SEED):
    if not LINGAM_AVAILABLE:
        raise ImportError("lingam package is required. Install it by `pip install lingam`.")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df.values)
    model = lingam.DirectLiNGAM(random_state=random_state)
    model.fit(X_scaled)
    B = pd.DataFrame(model.adjacency_matrix_, index=df.columns, columns=df.columns)

    edges = []
    for effect in B.index:
        for cause in B.columns:
            w = B.loc[effect, cause]
            if abs(w) >= threshold:
                edges.append({"Cause": cause, "Effect": effect, "Weight": w, "AbsWeight": abs(w)})
    edge_df = pd.DataFrame(edges).sort_values("AbsWeight", ascending=False)
    edge_df.to_csv(OUT_DIR / "Table_DirectLiNGAM_edges.csv", index=False)
    B.to_csv(OUT_DIR / "Matrix_DirectLiNGAM_adjacency.csv")
    return model, B, edge_df

lingam_model, B, edge_df = fit_direct_lingam(data, threshold=0.05)
display(edge_df.head(30))


def plot_directed_graph(edge_df, target=TARGET, max_edges=None):
    plot_edges = edge_df.copy()
    if max_edges is not None:
        plot_edges = plot_edges.head(max_edges)

    G = nx.DiGraph()
    for _, r in plot_edges.iterrows():
        G.add_edge(r["Cause"], r["Effect"], weight=r["Weight"], absweight=abs(r["Weight"]))

    fig, ax = plt.subplots(figsize=(13.5, 10.5))
    pos = nx.spring_layout(G, seed=SEED, k=1.2)

    node_colours = []
    for node in G.nodes:
        if node == target:
            node_colours.append(PALETTE["red"])
        elif node in TREATMENTS:
            node_colours.append(PALETTE["orange"])
        else:
            node_colours.append(PALETTE["cyan"])

    weights = np.array([G[u][v]["absweight"] for u, v in G.edges])
    widths = 0.8 + 4.5 * (weights - weights.min()) / (weights.max() - weights.min() + 1e-12)
    edge_colours = [PALETTE["blue"] if G[u][v]["weight"] > 0 else PALETTE["purple"] for u, v in G.edges]

    nx.draw_networkx_edges(G, pos, ax=ax, width=widths, edge_color=edge_colours,
                           alpha=0.35, arrows=True, arrowsize=14, arrowstyle="-|>",
                           connectionstyle="arc3,rad=0.08")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=900, node_color=node_colours,
                           edgecolors="white", linewidths=1.5, alpha=0.98)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_family="Times New Roman")

    patches = [
        mpatches.Patch(color=PALETTE["red"], label="Outcome"),
        mpatches.Patch(color=PALETTE["orange"], label="Treatment variables"),
        mpatches.Patch(color=PALETTE["cyan"], label="Other observed covariates"),
        mpatches.Patch(color=PALETTE["blue"], label="Positive edge"),
        mpatches.Patch(color=PALETTE["purple"], label="Negative edge")
    ]
    ax.legend(handles=patches, loc="lower left", frameon=True, edgecolor="#D0D5DD")
    ax.set_title("DirectLiNGAM directional-dependence network")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DirectLiNGAM_directional_network.png", bbox_inches="tight")
    plt.show()
    return G

G_lingam = plot_directed_graph(edge_df, max_edges=None)

In [ ]:
# ============================================================
# Module 5. Acyclicity diagnostics
# ============================================================
def acyclicity_diagnostics(edge_df):
    G = nx.DiGraph()
    G.add_nodes_from(data.columns)
    for _, r in edge_df.iterrows():
        G.add_edge(r["Cause"], r["Effect"], weight=r["Weight"])
    is_dag = nx.is_directed_acyclic_graph(G)
    topo_order = list(nx.topological_sort(G)) if is_dag else None
    cycles = [] if is_dag else list(nx.simple_cycles(G))
    summary = pd.DataFrame({
        "Diagnostic": ["Is directed acyclic graph", "Number of nodes", "Number of retained edges", "Number of directed cycles"],
        "Result": [is_dag, G.number_of_nodes(), G.number_of_edges(), len(cycles)]
    })
    summary.to_csv(OUT_DIR / "Table_DirectLiNGAM_acyclicity_summary.csv", index=False)
    if topo_order is not None:
        pd.DataFrame({"Causal order": topo_order}).to_csv(OUT_DIR / "Table_DirectLiNGAM_topological_order.csv", index=False)
    display(summary)
    if topo_order is not None:
        print("Topological order:")
        print(" -> ".join(topo_order))
    else:
        print("Directed cycles detected:", cycles[:10])
    return G, summary, topo_order, cycles

G_dag, dag_summary, topo_order, cycles = acyclicity_diagnostics(edge_df)

In [ ]:
# ============================================================
# Module 6. Linearity approximation diagnostics
# ============================================================
def get_parents_from_B(B, node, threshold=0.05):
    if node not in B.index:
        return []
    parents = B.columns[(B.loc[node].abs() >= threshold)].tolist()
    parents = [p for p in parents if p != node]
    return parents


def linearity_diagnostics(df, B, threshold=0.05, cv_splits=5):
    rows = []
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=SEED)
    for node in df.columns:
        parents = get_parents_from_B(B, node, threshold)
        if len(parents) == 0:
            rows.append({
                "Node": node, "No. parents": 0, "Parents": "",
                "Linear CV RMSE": np.nan, "RF CV RMSE": np.nan,
                "RF improvement over linear (%)": np.nan,
                "Interpretation": "Exogenous/no retained parent"
            })
            continue
        X = df[parents].values
        y = df[node].values
        linear_pipe = Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())])
        rf = RandomForestRegressor(n_estimators=600, min_samples_leaf=5, random_state=SEED, n_jobs=-1)
        linear_rmse = -cross_val_score(linear_pipe, X, y, cv=kf, scoring="neg_root_mean_squared_error").mean()
        rf_rmse = -cross_val_score(rf, X, y, cv=kf, scoring="neg_root_mean_squared_error").mean()
        improvement = 100 * (linear_rmse - rf_rmse) / (linear_rmse + 1e-12)
        if improvement < 5:
            interp = "Linear approximation acceptable"
        elif improvement < 15:
            interp = "Moderate non-linearity"
        else:
            interp = "Strong non-linearity; interpret DirectLiNGAM as exploratory"
        rows.append({
            "Node": node,
            "No. parents": len(parents),
            "Parents": "; ".join(parents),
            "Linear CV RMSE": linear_rmse,
            "RF CV RMSE": rf_rmse,
            "RF improvement over linear (%)": improvement,
            "Interpretation": interp
        })
    result = pd.DataFrame(rows).sort_values("RF improvement over linear (%)", ascending=False)
    result.to_csv(OUT_DIR / "Table_DirectLiNGAM_linearity_approximation.csv", index=False)
    return result

lin_table = linearity_diagnostics(data, B, threshold=0.05)
display(lin_table)


def plot_linearity_diagnostics(lin_table):
    plot_df = lin_table.dropna(subset=["RF improvement over linear (%)"]).copy()
    plot_df = plot_df.sort_values("RF improvement over linear (%)", ascending=True)
    fig, ax = plt.subplots(figsize=(9.5, max(5.5, 0.35 * len(plot_df))))
    colours = [PALETTE["red"] if v >= 15 else PALETTE["orange"] if v >= 5 else PALETTE["teal"]
               for v in plot_df["RF improvement over linear (%)"]]
    ax.barh(plot_df["Node"], plot_df["RF improvement over linear (%)"], color=colours, edgecolor="white", linewidth=0.6)
    ax.axvline(5, color=PALETTE["orange"], ls="--", lw=1.2, label="Moderate threshold (5%)")
    ax.axvline(15, color=PALETTE["red"], ls="--", lw=1.2, label="Strong threshold (15%)")
    ax.set_xlabel("RMSE reduction of Random Forest over linear model (%)")
    ax.set_ylabel("")
    ax.set_title("Linearity approximation diagnostic for DirectLiNGAM structural equations")
    ax.grid(axis="x", color="#E9EDF2", linewidth=0.8)
    ax.legend(frameon=True, edgecolor="#D0D5DD")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DirectLiNGAM_linearity_diagnostics.png", bbox_inches="tight")
    plt.show()

plot_linearity_diagnostics(lin_table)

In [ ]:
# ============================================================
# Module 7. Residual independence diagnostics
# ============================================================
def structural_residuals(df, B, threshold=0.05):
    residuals = pd.DataFrame(index=df.index)
    fitted_info = []
    for node in df.columns:
        parents = get_parents_from_B(B, node, threshold)
        y = df[node].values
        if len(parents) == 0:
            residuals[node] = y - np.mean(y)
            fitted_info.append({"Node": node, "No. parents": 0, "In-sample R2": np.nan})
        else:
            X = df[parents].values
            model = Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())])
            model.fit(X, y)
            pred = model.predict(X)
            residuals[node] = y - pred
            fitted_info.append({"Node": node, "No. parents": len(parents), "In-sample R2": r2_score(y, pred)})
    info = pd.DataFrame(fitted_info)
    return residuals, info

resid_df, resid_info = structural_residuals(data, B, threshold=0.05)
display(resid_info)
resid_info.to_csv(OUT_DIR / "Table_DirectLiNGAM_structural_residual_model_fit.csv", index=False)


def residual_independence_table(resid_df, alpha=0.05):
    rows = []
    cols = resid_df.columns.tolist()
    corr_mat = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)
    p_mat = pd.DataFrame(np.zeros((len(cols), len(cols))), index=cols, columns=cols)
    for a, b in combinations(cols, 2):
        r, p = spearmanr(resid_df[a], resid_df[b])
        corr_mat.loc[a, b] = corr_mat.loc[b, a] = r
        p_mat.loc[a, b] = p_mat.loc[b, a] = p
        rows.append({"Residual A": a, "Residual B": b, "Spearman rho": r, "p-value": p})
    pair_df = pd.DataFrame(rows).sort_values("p-value")
    pair_df["Significant at 0.05"] = pair_df["p-value"] < alpha
    pair_df.to_csv(OUT_DIR / "Table_DirectLiNGAM_residual_independence_pairwise.csv", index=False)
    corr_mat.to_csv(OUT_DIR / "Matrix_DirectLiNGAM_residual_correlation.csv")
    p_mat.to_csv(OUT_DIR / "Matrix_DirectLiNGAM_residual_independence_pvalues.csv")
    return pair_df, corr_mat, p_mat

resid_ind_table, resid_corr, resid_p = residual_independence_table(resid_df)
display(resid_ind_table.head(20))


def plot_residual_correlation_heatmap(corr_mat):
    fig, ax = plt.subplots(figsize=(11, 9.5))
    cmap = LinearSegmentedColormap.from_list("resid_cmap", ["#4F6D9A", "#F8FAFC", "#B6564C"])
    im = ax.imshow(corr_mat.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(np.arange(len(corr_mat.columns)))
    ax.set_yticks(np.arange(len(corr_mat.index)))
    ax.set_xticklabels(corr_mat.columns, rotation=75, ha="right", fontsize=8)
    ax.set_yticklabels(corr_mat.index, fontsize=8)
    ax.set_title("Pairwise Spearman correlation among structural residuals")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Spearman rho")
    ax.set_xticks(np.arange(-.5, len(corr_mat.columns), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(corr_mat.index), 1), minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DirectLiNGAM_residual_independence_heatmap.png", bbox_inches="tight")
    plt.show()

plot_residual_correlation_heatmap(resid_corr)

In [ ]:
# ============================================================
# Module 8. Temporal residual diagnostics
# ============================================================
def temporal_residual_diagnostics(resid_df, lags=(1, 3, 7, 14), alpha=0.05):
    if not STATSMODELS_AVAILABLE:
        print("statsmodels is unavailable; skipping temporal diagnostics.")
        return pd.DataFrame()
    rows = []
    for col in resid_df.columns:
        x = resid_df[col].dropna().values
        dw = durbin_watson(x)
        lb = acorr_ljungbox(x, lags=list(lags), return_df=True)
        for lag in lags:
            rows.append({
                "Residual variable": col,
                "Lag": lag,
                "Ljung-Box statistic": lb.loc[lag, "lb_stat"],
                "Ljung-Box p-value": lb.loc[lag, "lb_pvalue"],
                "Durbin-Watson": dw,
                "Temporal dependence indicated": lb.loc[lag, "lb_pvalue"] < alpha
            })
    result = pd.DataFrame(rows)
    result.to_csv(OUT_DIR / "Table_DirectLiNGAM_temporal_residual_diagnostics.csv", index=False)
    return result

temp_resid_table = temporal_residual_diagnostics(resid_df)
display(temp_resid_table.head(30))


def plot_temporal_diagnostics(temp_table):
    if temp_table.empty:
        return
    pivot = temp_table.pivot(index="Residual variable", columns="Lag", values="Ljung-Box p-value")
    plot_vals = -np.log10(pivot.clip(lower=1e-12))
    fig, ax = plt.subplots(figsize=(8.5, max(5.5, 0.35 * len(plot_vals))))
    im = ax.imshow(plot_vals.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(np.arange(plot_vals.shape[1]))
    ax.set_yticks(np.arange(plot_vals.shape[0]))
    ax.set_xticklabels([f"Lag {c}" for c in plot_vals.columns])
    ax.set_yticklabels(plot_vals.index, fontsize=8)
    ax.set_title("Temporal dependence diagnostic: -log10(Ljung-Box p-value)")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("-log10(p-value)")
    ax.set_xticks(np.arange(-.5, plot_vals.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, plot_vals.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DirectLiNGAM_temporal_residual_diagnostics.png", bbox_inches="tight")
    plt.show()

plot_temporal_diagnostics(temp_resid_table)

In [ ]:
# ============================================================
# Module 9. DirectLiNGAM graph stability diagnostics
# ============================================================
def make_block_indices(n, block_size=14, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    starts = np.arange(0, n)
    idx = []
    while len(idx) < n:
        s = int(rng.choice(starts))
        block = list(range(s, min(s + block_size, n)))
        if len(block) < block_size:
            block += list(range(0, block_size - len(block)))
        idx.extend(block)
    return np.array(idx[:n])


def lingam_edge_set(df_sample, threshold=0.05, random_state=SEED):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_sample.values)
    model = lingam.DirectLiNGAM(random_state=random_state)
    model.fit(X_scaled)
    B_sample = pd.DataFrame(model.adjacency_matrix_, index=df_sample.columns, columns=df_sample.columns)
    edges = set()
    for effect in B_sample.index:
        for cause in B_sample.columns:
            if abs(B_sample.loc[effect, cause]) >= threshold:
                edges.add((cause, effect))
    return edges


def graph_stability_bootstrap(df, n_boot=200, threshold=0.05, mode="iid", block_size=14):
    if not LINGAM_AVAILABLE:
        raise ImportError("lingam package is required.")
    rng = np.random.default_rng(SEED)
    n = len(df)
    edge_counts = {}
    all_edges = []
    for b in range(n_boot):
        if mode == "iid":
            idx = rng.choice(np.arange(n), size=n, replace=True)
        elif mode == "block":
            idx = make_block_indices(n, block_size=block_size, rng=rng)
        else:
            raise ValueError("mode must be 'iid' or 'block'.")
        sample = df.iloc[idx].reset_index(drop=True)
        try:
            edges_b = lingam_edge_set(sample, threshold=threshold, random_state=SEED + b)
        except Exception:
            continue
        all_edges.extend(list(edges_b))
        for e in edges_b:
            edge_counts[e] = edge_counts.get(e, 0) + 1
    out = pd.DataFrame([
        {"Cause": e[0], "Effect": e[1], "Frequency": c / n_boot, "Count": c, "Mode": mode}
        for e, c in edge_counts.items()
    ]).sort_values("Frequency", ascending=False)
    return out

# The default value is 200 iterations to reduce runtime. For final manuscript results, this can be increased to 500 or 1000.
iid_stability = graph_stability_bootstrap(data, n_boot=200, threshold=0.05, mode="iid")
block_stability = graph_stability_bootstrap(data, n_boot=200, threshold=0.05, mode="block", block_size=14)
stability_table = pd.concat([iid_stability, block_stability], ignore_index=True)
stability_table.to_csv(OUT_DIR / "Table_DirectLiNGAM_edge_stability_bootstrap.csv", index=False)
display(stability_table.head(30))


def plot_edge_stability(stability_table, top_n=25):
    if stability_table.empty:
        return
    tmp = stability_table.copy()
    tmp["Edge"] = tmp["Cause"] + " → " + tmp["Effect"]
    mean_freq = tmp.groupby("Edge")["Frequency"].mean().sort_values(ascending=False).head(top_n).index
    plot_df = tmp[tmp["Edge"].isin(mean_freq)].copy()
    order = plot_df.groupby("Edge")["Frequency"].mean().sort_values().index.tolist()

    fig, ax = plt.subplots(figsize=(10.5, max(6.5, 0.38 * len(order))))
    y_pos = np.arange(len(order))
    width = 0.36
    mode_to_offset = {"iid": -width/2, "block": width/2}
    mode_to_colour = {"iid": PALETTE["blue"], "block": PALETTE["orange"]}
    for mode in ["iid", "block"]:
        sub = plot_df[plot_df["Mode"] == mode].set_index("Edge").reindex(order)
        ax.barh(y_pos + mode_to_offset[mode], sub["Frequency"], height=width,
                color=mode_to_colour[mode], edgecolor="white", label=mode.upper())
    ax.axvline(0.50, color=PALETTE["grey"], lw=1.1, ls="--", label="50% frequency")
    ax.axvline(0.70, color=PALETTE["red"], lw=1.1, ls="--", label="70% frequency")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(order, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Bootstrap edge frequency")
    ax.set_title("DirectLiNGAM graph-stability diagnostic")
    ax.grid(axis="x", color="#E9EDF2", linewidth=0.8)
    ax.legend(frameon=True, edgecolor="#D0D5DD", loc="lower right")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DirectLiNGAM_edge_stability_bootstrap.png", bbox_inches="tight")
    plt.show()

plot_edge_stability(stability_table, top_n=25)

In [ ]:
# ============================================================
# Module 10. Adjustment-set construction for DML-CF robustness
# ============================================================
def parents_of(node, B, threshold=0.05):
    return get_parents_from_B(B, node, threshold)


def build_adjustment_sets(df, B, treatment, outcome=TARGET, threshold=0.05):
    all_covariates = [c for c in df.columns if c not in [treatment, outcome]]
    treatment_parents = parents_of(treatment, B, threshold)
    outcome_parents = parents_of(outcome, B, threshold)
    parent_only = sorted(set(treatment_parents + outcome_parents) - {treatment, outcome})
    domain_set = sorted(set(parent_only + [c for c in POTENTIAL_LATENT_PROXIES if c in df.columns]) - {treatment, outcome})
    all_observed = all_covariates
    sets = {
        "parent_only": parent_only,
        "parent_plus_domain": domain_set,
        "all_observed_covariates": all_observed
    }
    return sets

adjustment_sets = {t: build_adjustment_sets(data, B, t) for t in TREATMENTS}
for t, sets in adjustment_sets.items():
    print("\nTreatment:", t)
    for name, covs in sets.items():
        print(f"  {name}: {len(covs)} covariates")
        print("   ", covs)

In [ ]:
# ============================================================
# Module 11. DML-CF estimation functions and robustness loop
# ============================================================
def estimate_ate_causal_forest(df, treatment, outcome, covariates,
                               cv_mode="kfold", n_estimators=1000,
                               min_samples_leaf=10, random_state=SEED):
    """Estimate ATE using EconML CausalForestDML for continuous treatments."""
    if not ECONML_AVAILABLE:
        raise ImportError("econml is required. Install it by `pip install econml`.")
    X = df[covariates].values if len(covariates) > 0 else None
    T = df[treatment].values
    Y = df[outcome].values

    if cv_mode == "kfold":
        cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
    elif cv_mode == "timeseries":
        cv = TimeSeriesSplit(n_splits=5)
    else:
        raise ValueError("cv_mode must be 'kfold' or 'timeseries'.")

    model_y = LassoCV(cv=5, random_state=random_state, max_iter=20000)
    model_t = LassoCV(cv=5, random_state=random_state, max_iter=20000)

    cf = CausalForestDML(
        model_y=model_y,
        model_t=model_t,
        discrete_treatment=False,
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        max_depth=None,
        cv=cv,
        random_state=random_state,
        n_jobs=-1
    )
    cf.fit(Y, T, X=X)
    effects = cf.effect(X)
    ate = float(np.mean(effects))
    return ate, effects, cf


def run_dml_robustness(df, treatments=TREATMENTS, outcome=TARGET, adjustment_sets=adjustment_sets,
                       n_estimators=1000):
    rows = []
    effect_store = {}
    for treatment in treatments:
        for set_name, covariates in adjustment_sets[treatment].items():
            for cv_mode in ["kfold", "timeseries"]:
                try:
                    ate, effects, model = estimate_ate_causal_forest(
                        df, treatment, outcome, covariates,
                        cv_mode=cv_mode, n_estimators=n_estimators,
                        random_state=SEED
                    )
                    rows.append({
                        "Treatment": treatment,
                        "Adjustment set": set_name,
                        "No. covariates": len(covariates),
                        "CV mode": cv_mode,
                        "ATE": ate,
                        "CATE std": float(np.std(effects)),
                        "CATE q05": float(np.quantile(effects, 0.05)),
                        "CATE q50": float(np.quantile(effects, 0.50)),
                        "CATE q95": float(np.quantile(effects, 0.95))
                    })
                    effect_store[(treatment, set_name, cv_mode)] = effects
                except Exception as e:
                    rows.append({
                        "Treatment": treatment,
                        "Adjustment set": set_name,
                        "No. covariates": len(covariates),
                        "CV mode": cv_mode,
                        "ATE": np.nan,
                        "CATE std": np.nan,
                        "CATE q05": np.nan,
                        "CATE q50": np.nan,
                        "CATE q95": np.nan,
                        "Error": str(e)
                    })
    result = pd.DataFrame(rows)
    result.to_csv(OUT_DIR / "Table_DML_CF_ATE_adjustment_set_robustness.csv", index=False)
    return result, effect_store

# If runtime is excessive, set n_estimators to 300 for preliminary testing; final manuscript results should use 1000 or higher.
dml_robust_table, dml_effect_store = run_dml_robustness(df=model_df, n_estimators=1000)
display(dml_robust_table)


def plot_dml_robustness(result):
    plot_df = result.dropna(subset=["ATE"]).copy()
    if plot_df.empty:
        return
    treatments = plot_df["Treatment"].unique().tolist()
    fig, axes = plt.subplots(1, len(treatments), figsize=(5.2 * len(treatments), 5.2), sharey=False)
    if len(treatments) == 1:
        axes = [axes]
    for ax, t in zip(axes, treatments):
        sub = plot_df[plot_df["Treatment"] == t].copy()
        sub["Label"] = sub["Adjustment set"] + "\n" + sub["CV mode"]
        x = np.arange(len(sub))
        colours = [PALETTE["blue"] if m == "kfold" else PALETTE["orange"] for m in sub["CV mode"]]
        ax.bar(x, sub["ATE"], color=colours, edgecolor="white", linewidth=0.8)
        ax.axhline(0, color=PALETTE["dark"], lw=1.0)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["Label"], rotation=45, ha="right", fontsize=8)
        ax.set_title(t)
        ax.set_ylabel("ATE")
        ax.grid(axis="y", color="#E9EDF2", linewidth=0.8)
    fig.suptitle("DML-CF ATE robustness across adjustment sets and validation schemes", y=1.02)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DML_CF_adjustment_set_robustness.png", bbox_inches="tight")
    plt.show()

plot_dml_robustness(dml_robust_table)

In [ ]:
# ============================================================
# Module 12. Temporal block bootstrap uncertainty for DML-CF
# ============================================================
def block_bootstrap_dml_ate(df, treatment, outcome, covariates,
                            n_boot=300, block_size=14, cv_mode="kfold",
                            n_estimators=500):
    rng = np.random.default_rng(SEED)
    n = len(df)
    ates = []
    for b in range(n_boot):
        idx = make_block_indices(n, block_size=block_size, rng=rng)
        sample = df.iloc[idx].reset_index(drop=True)
        try:
            ate, _, _ = estimate_ate_causal_forest(
                sample, treatment, outcome, covariates,
                cv_mode=cv_mode, n_estimators=n_estimators,
                random_state=SEED + b
            )
            ates.append(ate)
        except Exception:
            continue
    ates = np.array(ates, dtype=float)
    return ates

# The main analysis is recommended to use parent_plus_domain to balance graph-based structure and domain-informed confounding control; this can also be changed to all_observed_covariates.
BOOTSTRAP_SET_NAME = "parent_plus_domain"
BOOT_N = 300
BLOCK_SIZE = 14

boot_rows = []
boot_store = {}
for t in TREATMENTS:
    covs = adjustment_sets[t][BOOTSTRAP_SET_NAME]
    ates = block_bootstrap_dml_ate(data, t, TARGET, covs, n_boot=BOOT_N, block_size=BLOCK_SIZE, cv_mode="kfold", n_estimators=500)
    boot_store[t] = ates
    if len(ates) > 0:
        boot_rows.append({
            "Treatment": t,
            "Adjustment set": BOOTSTRAP_SET_NAME,
            "Block size": BLOCK_SIZE,
            "Successful bootstrap runs": len(ates),
            "ATE mean": float(np.mean(ates)),
            "ATE median": float(np.median(ates)),
            "95% CI lower": float(np.quantile(ates, 0.025)),
            "95% CI upper": float(np.quantile(ates, 0.975)),
            "Std": float(np.std(ates))
        })

boot_table = pd.DataFrame(boot_rows)
boot_table.to_csv(OUT_DIR / "Table_DML_CF_block_bootstrap_ATE.csv", index=False)
display(boot_table)


# def plot_block_bootstrap_distributions(boot_store):
#     n = len(boot_store)
#     fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.8), sharey=False)
#     if n == 1:
#         axes = [axes]
#     for ax, (t, ates) in zip(axes, boot_store.items()):
#         ates = np.asarray(ates)
#         if len(ates) == 0:
#             ax.set_title(t + "\n(no successful run)")
#             continue
#         ci = np.quantile(ates, [0.025, 0.975])
#         mean_ate = np.mean(ates)
#         ax.hist(ates, bins=30, density=True, color=PALETTE["cyan"], alpha=0.75,
#                 edgecolor="white", linewidth=0.5)
#         ax.axvline(mean_ate, color=PALETTE["dark"], lw=2.0, label=f"Mean = {mean_ate:.3f}")
#         ax.axvline(ci[0], color=PALETTE["red"], lw=1.5, ls="--", label=f"95% CI [{ci[0]:.3f}, {ci[1]:.3f}]")
#         ax.axvline(ci[1], color=PALETTE["red"], lw=1.5, ls="--")
#         ax.set_title(t)
#         ax.set_xlabel("ATE")
#         ax.set_ylabel("Density")
#         ax.grid(axis="y", color="#E9EDF2", linewidth=0.8)
#         ax.legend(frameon=True, edgecolor="#D0D5DD", fontsize=8)
#     fig.suptitle("Temporal block-bootstrap distributions of DML-CF ATEs", y=1.02)
#     fig.tight_layout()
#     fig.savefig(OUT_DIR / "Fig_DML_CF_block_bootstrap_ATE_distributions.png", bbox_inches="tight")
#     plt.show()

# plot_block_bootstrap_distributions(boot_store)

In [ ]:
def plot_block_bootstrap_distributions(boot_store):
    n = len(boot_store)

    fig, axes = plt.subplots(
        1, n,
        figsize=(5.0 * n, 4.8),
        sharey=False
    )

    if n == 1:
        axes = [axes]

    # ------------------------------------------------------------
    # Safe colour settings
    # ------------------------------------------------------------
    # The previous error occurred because PALETTE did not contain the key "cyan".
    # PALETTE["cyan"] is not used here. A refined blue-green hexadecimal colour is specified directly to avoid another KeyError.
    hist_color = "#4DBBD5"
    mean_color = PALETTE["dark"] if "dark" in PALETTE else "#222222"
    ci_color = PALETTE["red"] if "red" in PALETTE else "#D62728"
    grid_color = "#E9EDF2"

    for ax, (t, ates) in zip(axes, boot_store.items()):

        ates = np.asarray(ates, dtype=float)
        ates = ates[np.isfinite(ates)]

        if len(ates) == 0:
            ax.set_title(t + "\n(no successful run)", fontsize=11, fontweight="bold")
            ax.axis("off")
            continue

        ci = np.quantile(ates, [0.025, 0.975])
        mean_ate = np.mean(ates)
        median_ate = np.median(ates)

        # Histogram
        ax.hist(
            ates,
            bins=30,
            density=True,
            color=hist_color,
            alpha=0.75,
            edgecolor="white",
            linewidth=0.6
        )

        # Mean line
        ax.axvline(
            mean_ate,
            color=mean_color,
            lw=2.0,
            label=f"Mean = {mean_ate:.3f}"
        )

        # Median line
        ax.axvline(
            median_ate,
            color="#7E57C2",
            lw=1.6,
            ls=":",
            label=f"Median = {median_ate:.3f}"
        )

        # 95% CI lines
        ax.axvline(
            ci[0],
            color=ci_color,
            lw=1.5,
            ls="--",
            label=f"95% CI [{ci[0]:.3f}, {ci[1]:.3f}]"
        )

        ax.axvline(
            ci[1],
            color=ci_color,
            lw=1.5,
            ls="--"
        )

        # Zero reference line
        ax.axvline(
            0,
            color="#7F7F7F",
            lw=1.0,
            ls="-."
        )

        ax.set_title(t, fontsize=12, fontweight="bold")
        ax.set_xlabel("ATE", fontsize=11)
        ax.set_ylabel("Density", fontsize=11)

        ax.grid(axis="y", color=grid_color, linewidth=0.8)
        ax.set_axisbelow(True)

        ax.legend(
            frameon=True,
            edgecolor="#D0D5DD",
            fontsize=8,
            loc="best"
        )

        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)

    fig.suptitle(
        "Temporal block-bootstrap distributions of DML-CF ATEs",
        fontsize=14,
        fontweight="bold",
        y=1.03
    )

    fig.tight_layout()

    fig.savefig(
        OUT_DIR / "Fig_DML_CF_block_bootstrap_ATE_distributions.png",
        bbox_inches="tight",
        dpi=600
    )

    plt.show()


plot_block_bootstrap_distributions(boot_store)

In [ ]:
# ============================================================
# Module 13. Leave-one-covariate-out sensitivity
# ============================================================
def leave_one_covariate_out_sensitivity(df, treatment, outcome, base_covariates,
                                        n_estimators=500, cv_mode="kfold"):
    rows = []
    base_ate, _, _ = estimate_ate_causal_forest(
        df, treatment, outcome, base_covariates,
        cv_mode=cv_mode, n_estimators=n_estimators, random_state=SEED
    )
    rows.append({
        "Treatment": treatment,
        "Removed covariate": "None (base)",
        "No. covariates": len(base_covariates),
        "ATE": base_ate,
        "ATE change from base": 0.0,
        "Relative change (%)": 0.0
    })
    for c in base_covariates:
        covs = [x for x in base_covariates if x != c]
        try:
            ate, _, _ = estimate_ate_causal_forest(
                df, treatment, outcome, covs,
                cv_mode=cv_mode, n_estimators=n_estimators, random_state=SEED
            )
            rows.append({
                "Treatment": treatment,
                "Removed covariate": c,
                "No. covariates": len(covs),
                "ATE": ate,
                "ATE change from base": ate - base_ate,
                "Relative change (%)": 100 * (ate - base_ate) / (abs(base_ate) + 1e-12)
            })
        except Exception as e:
            rows.append({
                "Treatment": treatment,
                "Removed covariate": c,
                "No. covariates": len(covs),
                "ATE": np.nan,
                "ATE change from base": np.nan,
                "Relative change (%)": np.nan,
                "Error": str(e)
            })
    return pd.DataFrame(rows)

loo_tables = []
for t in TREATMENTS:
    covs = adjustment_sets[t]["parent_plus_domain"]
    if len(covs) >= 2:
        loo_tables.append(leave_one_covariate_out_sensitivity(data, t, TARGET, covs, n_estimators=500))
loo_table = pd.concat(loo_tables, ignore_index=True) if loo_tables else pd.DataFrame()
loo_table.to_csv(OUT_DIR / "Table_DML_CF_leave_one_covariate_out_sensitivity.csv", index=False)
display(loo_table)


def plot_leave_one_out(loo_table, top_n=12):
    if loo_table.empty:
        return
    treatments = loo_table["Treatment"].unique().tolist()
    fig, axes = plt.subplots(1, len(treatments), figsize=(5.5 * len(treatments), 5.5), sharey=False)
    if len(treatments) == 1:
        axes = [axes]
    for ax, t in zip(axes, treatments):
        sub = loo_table[(loo_table["Treatment"] == t) & (loo_table["Removed covariate"] != "None (base)")].dropna()
        sub = sub.reindex(sub["Relative change (%)"].abs().sort_values(ascending=False).index).head(top_n)
        sub = sub.sort_values("Relative change (%)")
        colours = [PALETTE["red"] if abs(v) >= 20 else PALETTE["orange"] if abs(v) >= 10 else PALETTE["blue"]
                   for v in sub["Relative change (%)"]]
        ax.barh(sub["Removed covariate"], sub["Relative change (%)"], color=colours, edgecolor="white")
        ax.axvline(0, color=PALETTE["dark"], lw=1.0)
        ax.axvline(10, color=PALETTE["orange"], ls="--", lw=1.0)
        ax.axvline(-10, color=PALETTE["orange"], ls="--", lw=1.0)
        ax.set_title(t)
        ax.set_xlabel("ATE change after removing one covariate (%)")
        ax.grid(axis="x", color="#E9EDF2", linewidth=0.8)
    fig.suptitle("Hidden-confounder proxy sensitivity: leave-one-covariate-out", y=1.02)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DML_CF_leave_one_covariate_out_sensitivity.png", bbox_inches="tight")
    plt.show()

plot_leave_one_out(loo_table)

In [ ]:
# ============================================================
# Module 14. Permutation placebo test
# ============================================================
def permutation_placebo_test(df, treatment, outcome, covariates,
                             n_perm=200, n_estimators=300):
    rng = np.random.default_rng(SEED)
    base_ate, _, _ = estimate_ate_causal_forest(
        df, treatment, outcome, covariates,
        cv_mode="kfold", n_estimators=n_estimators, random_state=SEED
    )
    placebo_ates = []
    for i in range(n_perm):
        df_perm = df.copy()
        df_perm[treatment] = rng.permutation(df_perm[treatment].values)
        try:
            ate_i, _, _ = estimate_ate_causal_forest(
                df_perm, treatment, outcome, covariates,
                cv_mode="kfold", n_estimators=n_estimators, random_state=SEED + i
            )
            placebo_ates.append(ate_i)
        except Exception:
            continue
    placebo_ates = np.asarray(placebo_ates)
    if len(placebo_ates) > 0:
        p_emp = (np.sum(np.abs(placebo_ates) >= abs(base_ate)) + 1) / (len(placebo_ates) + 1)
    else:
        p_emp = np.nan
    return base_ate, placebo_ates, p_emp

placebo_rows = []
placebo_store = {}
for t in TREATMENTS:
    covs = adjustment_sets[t]["parent_plus_domain"]
    try:
        base_ate, placebo_ates, p_emp = permutation_placebo_test(data, t, TARGET, covs, n_perm=200, n_estimators=300)
        placebo_store[t] = (base_ate, placebo_ates)
        placebo_rows.append({
            "Treatment": t,
            "Observed ATE": base_ate,
            "Placebo mean": float(np.mean(placebo_ates)) if len(placebo_ates) else np.nan,
            "Placebo 2.5%": float(np.quantile(placebo_ates, 0.025)) if len(placebo_ates) else np.nan,
            "Placebo 97.5%": float(np.quantile(placebo_ates, 0.975)) if len(placebo_ates) else np.nan,
            "Empirical two-sided p": p_emp,
            "Successful permutations": len(placebo_ates)
        })
    except Exception as e:
        placebo_rows.append({"Treatment": t, "Error": str(e)})

placebo_table = pd.DataFrame(placebo_rows)
placebo_table.to_csv(OUT_DIR / "Table_DML_CF_permutation_placebo_test.csv", index=False)
display(placebo_table)


def plot_placebo_distributions(placebo_store):
    n = len(placebo_store)
    fig, axes = plt.subplots(1, n, figsize=(5.0 * n, 4.8), sharey=False)
    if n == 1:
        axes = [axes]
    for ax, (t, (base_ate, placebo_ates)) in zip(axes, placebo_store.items()):
        ax.hist(placebo_ates, bins=30, density=True, color=PALETTE["light_grey"],
                edgecolor="white", linewidth=0.6, label="Placebo ATE")
        ax.axvline(base_ate, color=PALETTE["red"], lw=2.0, label=f"Observed ATE = {base_ate:.3f}")
        ax.axvline(0, color=PALETTE["dark"], lw=1.0)
        ax.set_title(t)
        ax.set_xlabel("ATE")
        ax.set_ylabel("Density")
        ax.grid(axis="y", color="#E9EDF2", linewidth=0.8)
        ax.legend(frameon=True, edgecolor="#D0D5DD", fontsize=8)
    fig.suptitle("Permutation placebo diagnostics for DML-CF ATE estimates", y=1.02)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "Fig_DML_CF_permutation_placebo_test.png", bbox_inches="tight")
    plt.show()

plot_placebo_distributions(placebo_store)

In [ ]:
# ============================================================
# Module 15. Automatic summary table
# ============================================================
def build_diagnostic_summary(ng_table, dag_summary, lin_table, resid_ind_table, temp_table, stability_table, dml_robust_table):
    rows = []
    rows.append({
        "Assumption / concern": "Non-Gaussianity",
        "Diagnostic implemented": "Shapiro-Wilk, Jarque-Bera, skewness and kurtosis",
        "Key result": f"{int(ng_table['Non-Gaussian by either test'].sum())}/{len(ng_table)} variables reject Gaussianity by either test.",
        "Recommended interpretation": "Supports the use of DirectLiNGAM, but non-Gaussianity alone is not sufficient for causal identification."
    })
    is_dag = dag_summary.loc[dag_summary['Diagnostic'] == 'Is directed acyclic graph', 'Result'].iloc[0]
    rows.append({
        "Assumption / concern": "Acyclicity",
        "Diagnostic implemented": "NetworkX directed-cycle check and topological sorting",
        "Key result": f"DAG check result: {is_dag}.",
        "Recommended interpretation": "The estimated graph satisfies the acyclic graph requirement at the retained-edge threshold."
    })
    strong_nonlin = lin_table[lin_table["RF improvement over linear (%)"] >= 15].shape[0]
    rows.append({
        "Assumption / concern": "Linearity",
        "Diagnostic implemented": "Cross-validated linear-vs-RF structural-equation comparison",
        "Key result": f"{strong_nonlin} nodes show strong non-linearity by the 15% RMSE-improvement rule.",
        "Recommended interpretation": "DirectLiNGAM should be interpreted as a linear directional-dependence approximation rather than a full nonlinear physical model."
    })
    sig_resid = int(resid_ind_table["Significant at 0.05"].sum()) if not resid_ind_table.empty else 0
    rows.append({
        "Assumption / concern": "Independent errors",
        "Diagnostic implemented": "Pairwise Spearman independence screening among structural residuals",
        "Key result": f"{sig_resid} residual pairs are significant at p < 0.05 before multiple-testing adjustment.",
        "Recommended interpretation": "Residual independence is diagnostically assessed, not proven; remaining dependence should be acknowledged."
    })
    if temp_table is not None and not temp_table.empty:
        dep_share = temp_table["Temporal dependence indicated"].mean()
        temp_key = f"{dep_share:.1%} of variable-lag tests indicate temporal dependence."
    else:
        temp_key = "Temporal diagnostics not available."
    rows.append({
        "Assumption / concern": "Repeated daily observations and temporal dependence",
        "Diagnostic implemented": "Ljung-Box and Durbin-Watson tests on structural residuals",
        "Key result": temp_key,
        "Recommended interpretation": "Findings should be interpreted at the daily operational scale; temporal dependence is handled as a diagnostic limitation and through block-bootstrap sensitivity."
    })
    if not stability_table.empty:
        stable_edges = stability_table[stability_table["Frequency"] >= 0.70].shape[0]
        stability_key = f"{stable_edges} edges appear with frequency >= 0.70 across bootstrap runs."
    else:
        stability_key = "Graph stability not available."
    rows.append({
        "Assumption / concern": "Graph stability",
        "Diagnostic implemented": "iid bootstrap and temporal block bootstrap edge-frequency analysis",
        "Key result": stability_key,
        "Recommended interpretation": "Stable edges can be emphasised; low-frequency edges should be treated as exploratory."
    })
    if dml_robust_table is not None and not dml_robust_table.empty:
        n_models = dml_robust_table["ATE"].notna().sum()
        dml_key = f"{n_models} DML-CF robustness specifications were successfully estimated."
    else:
        dml_key = "DML-CF robustness results not available."
    rows.append({
        "Assumption / concern": "Causal sufficiency / hidden confounders",
        "Diagnostic implemented": "Parent-only, parent-plus-domain, all-observed adjustment sets; leave-one-covariate-out; placebo test",
        "Key result": dml_key,
        "Recommended interpretation": "Causal effects should be reported as conditional on retained observed covariates, not as definitive proof under complete confounder control."
    })
    summary = pd.DataFrame(rows)
    summary.to_csv(OUT_DIR / "Table_Assumption_and_robustness_summary.csv", index=False)
    return summary

summary_table = build_diagnostic_summary(ng_table, dag_summary, lin_table, resid_ind_table, temp_resid_table, stability_table, dml_robust_table)
display(summary_table)
print(f"All outputs have been saved to: {OUT_DIR.resolve()}")